# CGT Evaluation Methodology

SynGraphBench evaluates CGT-generated synthetic graph data by training GNN classifiers on synthetic computation graph trees and testing them on original data. This follows the **train-on-synthetic, test-on-real** (TSTR) paradigm: a generative model is useful if downstream models trained on its output perform comparably to those trained on real data. Because CGT produces computation graph sequences rather than full graphs, the evaluation GNN must also operate on computation graph trees — requiring careful alignment of data representation, feature space, and batching between the synthetic and original conditions.

**Overview:**
1. Evaluation philosophy — train synthetic, test original
2. Three evaluation conditions and their roles
3. Computation graph construction for evaluation
4. Efficient batching via template edges
5. Feature normalization for fair comparison
6. Anomaly detection evaluation
7. Link prediction evaluation
8. The `num_layers = cg_depth` constraint
9. Usage

## 1. Evaluation Philosophy: Train Synthetic, Test Original

The core evaluation question is: *can a GNN trained entirely on CGT-generated synthetic data perform as well as one trained on real data?* This is the standard TSTR (Train on Synthetic, Test on Real) paradigm (Esteban et al., 2017). If the generative model has faithfully captured the data distribution, the performance gap should be small.

For CGT specifically, the generated data consists of cluster ID sequences representing computation graph trees — not a full graph with explicit edges. This means the evaluation GNN must also operate on computation graph trees: each node is classified based on its own rooted tree, not via message passing over the entire graph. We use the same GADBench GNN architectures (GCN, GIN, GraphSAGE) across all conditions — the only difference is how data is presented to them (batched trees vs. full graph).

Two downstream tasks are evaluated:
- **Anomaly detection** — binary node classification (normal vs. anomalous)
- **Link prediction** — scoring whether an edge exists between two nodes

Both tasks follow the same TSTR structure: train/validate on synthetic data, test on held-out original data.

## 2. Three Evaluation Conditions

A naïve comparison of "full-graph GNN on original data" vs. "computation-graph GNN on synthetic data" would conflate two effects: the quality of the synthetic features and the information loss from switching to a computation graph representation. To disentangle these, we evaluate under three conditions:

| Condition | Train/Val Data | Test Data | Purpose |
|---|---|---|---|
| **Original** | Original graph (full message passing) | Original graph (full) | Upper bound: best achievable performance |
| **Original-CG** | Original features in CG trees | Original features in CG trees | Control: isolates the effect of the CG representation |
| **Synthetic-CGT** | CGT cluster centre sequences in CG trees | Original features in CG trees | Target: measures CGT synthetic data utility |

**Why three conditions?** The Original-CG condition is the key control. The gap between *Original* and *Original-CG* measures how much information is lost by moving from full-graph message passing to fixed-depth computation graph trees. The gap between *Original-CG* and *Synthetic-CGT* measures the quality of CGT's generated features in isolation. Without Original-CG, we could not distinguish representation loss from generation loss.

Note that in both CG conditions, the **test set is always built from original features** (L2-normalized). In the Synthetic-CGT condition, only the train/val sets use CGT-generated cluster centre features. This ensures the test signal is consistent across conditions.

Implementation references:
- Original: `evaluate_models()` in `scripts/benchmark/anomaly_benchmark.py`
- Original-CG: `build_original_cg_datasets()` in `scripts/benchmark/bench_utils.py`
- Synthetic-CGT: `build_cgt_datasets()` in `scripts/benchmark/bench_utils.py`

In [ ]:
# Illustrative data flow for each evaluation condition

conditions = {
    "Original": {
        "train": "original_graph  -->  full GNN forward pass  -->  all node logits  -->  train loss",
        "test":  "original_graph  -->  full GNN forward pass  -->  all node logits  -->  metrics",
    },
    "Original-CG": {
        "train": "original_features (L2-normed)  -->  neighbor sampling  -->  CG trees  -->  GNN  -->  root logits  -->  train loss",
        "test":  "original_features (L2-normed)  -->  neighbor sampling  -->  CG trees  -->  GNN  -->  root logits  -->  metrics",
    },
    "Synthetic-CGT": {
        "train": "cluster_centers (unit-norm)[gen_ids]  -->  CG trees (fixed topology)  -->  GNN  -->  root logits  -->  train loss",
        "test":  "original_features (L2-normed)  -->  neighbor sampling  -->  CG trees  -->  GNN  -->  root logits  -->  metrics",
    },
}

for name, flows in conditions.items():
    print(f"=== {name} ===")
    for split, flow in flows.items():
        print(f"  {split:5s}: {flow}")
    print()

## 3. Computation Graph Construction for Evaluation

Both the Original-CG and Synthetic-CGT conditions present data to the GNN as **computation graph trees** — rooted ego-networks of fixed depth and fanout. Two dataset classes handle this, one for each data source.

### 3.1 Original features: `OriginalCompGraphDataset`

For each target node (seed), the dataset samples a fixed-structure tree of neighbours from the original graph's adjacency list (implemented in `GADBench/data/comp_graph.py`):

1. Start with the seed node as the root
2. For each of `step_num` (= `cg_depth`) hops, sample exactly `sample_num` (= `cg_fanout`) neighbours per node
3. If a node has fewer neighbours than `sample_num`, pad with `empty_id` — a special index pointing to a zero-feature row appended to the feature matrix
4. Optionally add `noise_num` random noise neighbours per node
5. Collect all sampled node IDs into a flat list and look up their features

The output is `{feat: [num_tree_nodes, feat_dim], label: [1]}`, where `num_tree_nodes = 1 + sample_num + sample_num² + ... = (sample_numᵈᵉᵖᵗʰ⁺¹ - 1) / (sample_num - 1)`.

### 3.2 Synthetic features: `SyntheticCompGraphDataset`

The CGT `.pt` file contains pre-generated cluster ID sequences (one per node) and a cluster centre matrix. No neighbour sampling is needed — the sequence *is* the computation graph, already in tree order. The dataset simply indexes `cluster_centers[sequence[i]]` to obtain the feature tensor.

The output has the **same shape** as `OriginalCompGraphDataset`: `{feat: [num_tree_nodes, feat_dim], label: [1]}`. This structural equivalence is what makes the comparison fair — the GNN sees identically shaped input regardless of whether the features come from the original graph or from CGT generation.

In [ ]:
import numpy as np

# Illustrate computation graph construction from original graph data

np.random.seed(42)
cg_depth = 2
cg_fanout = 3
feat_dim = 4

# Small graph: 8 nodes with adjacency lists
adjacency = {
    0: [1, 2, 3, 4],
    1: [0, 5],
    2: [0, 3, 6],
    3: [0, 2, 7],
    4: [0],
    5: [1],
    6: [2],
    7: [3],
}
features = np.random.randn(8, feat_dim).astype(np.float32)

# Pad features with a zero row for empty_id
empty_id = 8
features_padded = np.concatenate([features, np.zeros((1, feat_dim), dtype=np.float32)])

# Build computation graph for node 0
seed = 0
sampled = [seed]
curr_targets = [seed]

print(f"Seed node: {seed}")
print(f"cg_depth={cg_depth}, cg_fanout={cg_fanout}\n")

for hop in range(cg_depth):
    new_targets = []
    for tid in curr_targets:
        if tid == empty_id:
            neighbors = []
        else:
            neighbors = adjacency.get(tid, [])

        if len(neighbors) == 0:
            picked = [empty_id] * cg_fanout
        elif len(neighbors) < cg_fanout:
            picked = neighbors + [empty_id] * (cg_fanout - len(neighbors))
        else:
            picked = np.random.choice(neighbors, cg_fanout, replace=False).tolist()

        sampled.extend(picked)
        new_targets.extend(picked)

    print(f"Hop {hop+1}: sampled {len(new_targets)} nodes → {new_targets}")
    curr_targets = new_targets

feat_tensor = features_padded[sampled]
print(f"\nTree node IDs ({len(sampled)} total): {sampled}")
print(f"Feature tensor shape: {feat_tensor.shape}  (num_tree_nodes, feat_dim)")
print(f"\nExpected num_tree_nodes = 1 + {cg_fanout} + {cg_fanout}^2 = {1 + cg_fanout + cg_fanout**2}")

In [ ]:
import torch

# Illustrate SyntheticCompGraphDataset: cluster centre lookup

cluster_num = 8
feat_dim = 4
cg_fanout = 3
cg_depth = 2
num_tree_nodes = 1 + cg_fanout + cg_fanout ** 2  # 13

# Simulated cluster centres (from k-means during CGT training)
cluster_centers = torch.randn(cluster_num, feat_dim)
# Append zero row for empty_id
cluster_centers_padded = torch.cat([cluster_centers, torch.zeros(1, feat_dim)])

# A generated sequence from CGT: one per node, length = num_tree_nodes
# Each entry is a cluster ID (0..cluster_num-1) or empty_id (=cluster_num)
sequence = torch.tensor([3, 1, 5, 7, 0, 2, 8, 8, 4, 6, 1, 3, 8])
#                        ^root  ^hop-1 (3 nodes)  ^hop-2 (9 nodes, some empty)

# Feature reconstruction: simply index into cluster centres
feat_tensor = cluster_centers_padded[sequence]

print(f"Cluster centres shape: {cluster_centers.shape}  (cluster_num, feat_dim)")
print(f"Generated sequence:    {sequence.tolist()}")
print(f"Feature tensor shape:  {feat_tensor.shape}  (num_tree_nodes, feat_dim)")
print(f"\nSame shape as OriginalCompGraphDataset output: "
      f"({num_tree_nodes}, {feat_dim}) ✓")
print(f"\nEmpty nodes (id={cluster_num}) map to zero features:")
print(f"  feat_tensor[6] = {feat_tensor[6].tolist()}  (empty)")
print(f"  feat_tensor[0] = {feat_tensor[0].tolist()[:2]}...  (real cluster)")

## 4. Efficient Batching via Template Edges

All computation graphs share the same tree topology — determined entirely by `cg_depth` and `cg_fanout`. This invariance enables a significant batching optimization: instead of constructing a separate DGL graph per sample and calling `dgl.batch()`, we pre-compute the tree's edge list once and replicate it with offsets for each sample in the batch.

**Steps** (implemented in `GADBench/data/comp_graph.py`):

1. **`compute_tree_adj(step_num, sample_num)`** — builds the fixed tree adjacency matrix. Parent→child edges follow the breadth-first tree expansion.

2. **`compute_template_edges(tree_adj)`** — extracts edge arrays from the adjacency matrix. Crucially, edges are **reversed** to child→parent direction for GNN message passing (GNNs aggregate from neighbours *to* the target node). Self-loops are added if not already present.

3. **`make_comp_graph_collate(template_src, template_dst, num_tree_nodes)`** — returns a collate function that batches $B$ samples by:
   - Concatenating all feature tensors: $[B \times \text{num\_tree\_nodes}, \text{feat\_dim}]$
   - Offsetting the template edge indices by $i \times \text{num\_tree\_nodes}$ for sample $i$
   - Creating a single DGL graph with `set_batch_num_nodes()` and `set_batch_num_edges()`

This avoids the overhead of constructing $B$ separate DGL graphs. The offset trick works precisely because every tree has the same number of nodes and edges.

In [ ]:
import torch
from collections import defaultdict

# Illustrate template edge computation and batch collation

step_num = 2
sample_num = 3

# Step 1: Build fixed tree adjacency (same as compute_tree_adj)
sampled_nodes = [0]
curr_targets = [0]
edges = defaultdict(list)

for _ in range(step_num):
    new_targets = []
    for target in curr_targets:
        children = list(range(len(sampled_nodes), len(sampled_nodes) + sample_num))
        sampled_nodes.extend(children)
        new_targets.extend(children)
        edges[target].extend(children)
    curr_targets = new_targets

n = len(sampled_nodes)
print(f"Tree: {n} nodes (depth={step_num}, fanout={sample_num})")
print(f"  Level 0 (root): [0]")
print(f"  Level 1:        {list(range(1, 1 + sample_num))}")
print(f"  Level 2 (leaf): {list(range(1 + sample_num, n))}")

# Step 2: Extract edges — reversed for message passing (child → parent)
rows, cols = [], []
for parent, children in edges.items():
    for child in children:
        rows.append(child)   # src = child (reversed!)
        cols.append(parent)  # dst = parent

# Add self-loops
for i in range(n):
    rows.append(i)
    cols.append(i)

template_src = torch.tensor(rows)
template_dst = torch.tensor(cols)
num_edges = len(template_src)

print(f"\nTemplate edges ({num_edges} total, child→parent + self-loops):")
print(f"  src: {template_src.tolist()}")
print(f"  dst: {template_dst.tolist()}")

# Step 3: Batch collation with offsets (B=3 samples)
B = 3
offsets = torch.arange(B) * n

batch_src = (template_src.unsqueeze(0) + offsets.unsqueeze(1)).reshape(-1)
batch_dst = (template_dst.unsqueeze(0) + offsets.unsqueeze(1)).reshape(-1)

print(f"\nBatching {B} samples (each with {n} nodes):")
print(f"  Total nodes: {B * n}")
print(f"  Total edges: {B * num_edges}")
print(f"  Sample 0 node range: [0, {n-1}]")
print(f"  Sample 1 node range: [{n}, {2*n-1}]")
print(f"  Sample 2 node range: [{2*n}, {3*n-1}]")
print(f"\n  Batch src (first {num_edges} = sample 0): {batch_src[:num_edges].tolist()}")
print(f"  Batch src (next {num_edges} = sample 1):  {batch_src[num_edges:2*num_edges].tolist()}")

## 5. Feature Normalization for Fair Comparison

A single invariant runs through the whole evaluation pipeline: **every feature vector the GNN consumes has unit L2 norm**. Three pipeline stages enforce it explicitly so the synthetic and real feature pools share one common space.

1. **CGT training input** — `CGT/task/utils/utils.py` L2-normalizes the real feature matrix before k-means clustering and before GPT training.
2. **Cluster centres** — `cluster_feats` in `CGT/generator/cluster.py` L2-normalizes the cluster centres after k-means fits (and after the size-repair step), before they are saved into the `.pt` and indexed by downstream `SyntheticCompGraphDataset` / `build_synthetic_dgl_graph`. K-means centroids of L2-normalized vectors are already near-unit (mean ≈ 0.997), so this final step closes a small residual scale drift and converts an approximate property into an exact one. The appended `empty_id` row is intentionally left at zero norm — it represents a missing neighbour and is supposed to contribute no message-passing signal.
3. **Eval-time real features** — the downstream framework L2-normalizes the real-feature matrix it shows the GNN at test time:
   - Anomaly detection: `build_cgt_datasets()` and `build_original_cg_datasets()` in `scripts/benchmark/bench_utils.py`.
   - Link prediction: `evaluate_link_models_cgt()` in `scripts/benchmark/link_benchmark.py` normalizes `data.graph.ndata['feature']` once after loading, and the hybrid graph built by `build_synthetic_dgl_graph` inherits it.

**What happens without normalization?** Original features may have varying norms — some nodes have large feature vectors, others small. Cluster centres are produced by k-means on unit-norm inputs and so land near (but not exactly on) the unit sphere. If real-feature norms aren't aligned, the GNN sees two scales side-by-side: synthetic (unit-norm) and original (arbitrary norm). In link prediction this surfaces *inside a single forward pass* — a merged computation graph for one edge would mix unit-norm train/val nodes with raw-scale test/other nodes. The three-stage normalization removes that scale clash by construction and replaces an approximate property of k-means centroids with a single exact invariant that can be stated and defended in one sentence.

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

np.random.seed(7)

# Simulate node features with varying norms (as they appear in real datasets)
n_nodes = 6
feat_dim = 8
features = np.random.randn(n_nodes, feat_dim).astype(np.float32)
features[0] *= 10   # artificially large norm
features[3] *= 0.01  # artificially small norm

norms_before = np.linalg.norm(features, axis=1)
print("Before L2 normalization:")
print(f"  Norms: {norms_before.round(4)}")
print(f"  Range: [{norms_before.min():.4f}, {norms_before.max():.4f}]")

# Stage 3 in real code: build_cgt_datasets / build_original_cg_datasets (AD)
# and evaluate_link_models_cgt (LP) both apply this row-wise L2 norm.
features_normed = normalize(features, axis=1, norm='l2')

norms_after = np.linalg.norm(features_normed, axis=1)
print(f"\nAfter L2 normalization:")
print(f"  Norms: {norms_after.round(4)}")
print(f"  All unit norm: {np.allclose(norms_after, 1.0)}")

# Simulate cluster centres: k-means on L2-normalized features yields near-unit
# centroids; CGT/generator/cluster.py:cluster_feats then row-normalizes them
# explicitly so every saved centre lands exactly on the unit sphere.
raw_centers = np.random.randn(4, feat_dim).astype(np.float32) * 1.05  # slight scale drift
print(f"\nRaw k-means centre norms (near unit, not exact): "
      f"{np.linalg.norm(raw_centers, axis=1).round(4)}")
cluster_centers = normalize(raw_centers, axis=1, norm='l2')
center_norms = np.linalg.norm(cluster_centers, axis=1)
print(f"After cluster_feats L2-normalization: {center_norms.round(4)}")
print(f"  → Synthetic and original features now live in the same unit-sphere space")

## 6. Anomaly Detection Evaluation

Anomaly detection is the primary downstream task. The `CompGraphDetector` (in `GADBench/models/anomaly_detection/cgt_detector.py`) trains a standard GADBench GNN on batched computation graph trees and evaluates binary node classification (normal vs. anomalous).

### Architecture

The detector takes a batched DGL graph (produced by the template-edge collate function), runs a GNN forward pass to produce logits for every node in the batch, then extracts only the **root node logits** via `extract_root_logits()`. Root logits are used for classification.

**Why only root logits?** Each computation graph tree is rooted at the target node. After `cg_depth` layers of message passing, the root node's representation has aggregated information from its entire neighbourhood tree — exactly the information relevant for classifying that target node. Interior tree nodes have incomplete representations (they only see their subtree), so using them would be incorrect.

**`extract_root_logits()` mechanism:** Uses `batched_graph.batch_num_nodes()` to compute cumulative offsets. Node 0 of each sub-graph in the batch is the root, so root indices are `[0, num_tree_nodes, 2*num_tree_nodes, ...]`.

### Class Imbalance Handling

Anomaly detection datasets are heavily imbalanced (few anomalies, many normal nodes). `CompGraphDetector` computes `weight = num_neg / num_pos` from the training labels and uses weighted cross-entropy:

$$\mathcal{L} = \text{CrossEntropy}(\text{root\_logits}, \text{labels}, \text{weight}=[1.0, w])$$

This upweights the rare positive (anomaly) class proportionally to its scarcity.

### Early Stopping and Metrics

Training uses **early stopping on validation AUPRC** (area under precision-recall curve), which is more sensitive than AUROC for imbalanced data. The test scores at the best validation epoch are reported. Patience defaults to 50 epochs.

| Metric | Description | Why included |
|---|---|---|
| **AUROC** | Area under ROC curve | Standard ranking metric, widely reported |
| **AUPRC** | Area under precision-recall curve | Better for imbalanced data; used for early stopping |
| **Recall@K** | Recall among top-K scored nodes ($K$ = number of positive test nodes) | Practical: how many anomalies are found if we inspect the top-K predictions |

In [ ]:
import torch
import torch.nn.functional as F

# Illustrate root logit extraction and weighted cross-entropy loss

torch.manual_seed(0)

# Simulate a batch of 4 computation graph trees, each with 13 nodes
num_tree_nodes = 13
B = 4
total_nodes = B * num_tree_nodes

# Simulated GNN output: logits for ALL nodes in the batch (2 classes)
all_logits = torch.randn(total_nodes, 2)

# Extract root logits using cumulative offset (same as extract_root_logits)
batch_num_nodes = torch.full((B,), num_tree_nodes, dtype=torch.long)
root_ids = torch.zeros(B, dtype=torch.long)
root_ids[1:] = torch.cumsum(batch_num_nodes[:-1], dim=0)

root_logits = all_logits[root_ids]

print(f"Batch: {B} trees x {num_tree_nodes} nodes = {total_nodes} total nodes")
print(f"Root node indices: {root_ids.tolist()}")
print(f"Root logits shape: {root_logits.shape}  (B, num_classes)")

# Weighted cross-entropy for class imbalance
labels = torch.tensor([0, 0, 0, 1])  # 1 anomaly, 3 normal
num_pos = labels.sum().item()
num_neg = len(labels) - num_pos
weight = num_neg / max(num_pos, 1)

w = torch.tensor([1.0, weight])
loss = F.cross_entropy(root_logits, labels, weight=w)

print(f"\nLabels: {labels.tolist()} ({num_pos} pos, {num_neg} neg)")
print(f"Class weight: [1.0, {weight:.1f}]  (anomaly class upweighted {weight:.0f}x)")
print(f"Weighted CE loss: {loss.item():.4f}")

## 7. Link Prediction Evaluation

Link prediction evaluates whether the synthetic data preserves structural information — specifically, whether node embeddings learned from synthetic computation graphs can predict edges in the original graph. The `CompGraphLinkPredictor` (in `GADBench/models/link_prediction/cgt_link_predictor.py`) handles this.

### From Logits to Embeddings

The key difference from anomaly detection: instead of classification logits, the GNN outputs node **embeddings** (by passing `output_emb=True` to the GNN forward pass). Root embeddings are extracted the same way as root logits — via cumulative offset indexing. These embeddings are then used to score edges.

To score all edges efficiently, `CompGraphLinkPredictor._compute_all_embeddings()` builds an `OriginalCompGraphDataset` for **all** nodes in the graph (not just a split), batches them through the GNN, and collects root embeddings into a full matrix $\mathbf{H} \in \mathbb{R}^{N \times d}$. Edge $(u, v)$ is then scored using $\mathbf{h}_u$ and $\mathbf{h}_v$.

### Edge Decoders

Two decoders are available:

- **Dot product:** $\text{score}(u, v) = \mathbf{h}_u^\top \mathbf{h}_v$ — parameter-free, measures embedding similarity
- **MLP:** $\text{score}(u, v) = \text{MLP}(\mathbf{h}_u \odot \mathbf{h}_v)$ — learnable, better for capturing complex edge patterns. Architecture: `Linear → ReLU → Dropout → Linear(1)` on the Hadamard (element-wise) product.

### MST-Preserving Edge Split

The training graph must remain connected for GNN message passing to work properly. The `LinkDataset` class (in `GADBench/link_utils.py`) ensures this by computing a **minimum spanning tree** (MST) before splitting:

1. Compute the MST of the original graph using NetworkX
2. MST edges are **protected** — they always remain in the training set
3. Only non-MST edges are candidates for val/test splits
4. Candidates are split by shuffled permutation: test first, then val, remainder stays in train
5. The training graph is reconstructed without val/test edges via `dgl.remove_edges()`

### Negative Sampling

Each positive edge needs a corresponding negative (non-edge) for binary classification:

- **Random**: Uniform random node pairs, with hash-based collision detection to ensure they are true non-edges
- **Hard**: 2-hop random walks via `dgl.sampling.random_walk()` — the endpoint is structurally plausible (close in the graph) but not an actual neighbour, making the discrimination task harder and more informative

Val/test negatives are **fixed at dataset creation** for reproducible evaluation. Training negatives are **resampled each epoch** to prevent the model from memorising specific non-edges.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# Illustrate link prediction edge scoring with dot product and MLP decoders

n_nodes = 10
h_feats = 16

# Simulated root embeddings (output of GNN on computation graph trees)
h = torch.randn(n_nodes, h_feats)

# Positive edges (real edges) and negative edges (non-edges)
pos_src = torch.tensor([0, 1, 2, 3])
pos_dst = torch.tensor([1, 2, 3, 4])
neg_src = torch.tensor([0, 1, 2, 3])
neg_dst = torch.tensor([5, 6, 7, 8])

# --- Dot product decoder ---
pos_scores_dot = (h[pos_src] * h[pos_dst]).sum(dim=-1)
neg_scores_dot = (h[neg_src] * h[neg_dst]).sum(dim=-1)

print("=== Dot Product Decoder ===")
print(f"Positive edge scores: {pos_scores_dot.detach().numpy().round(3)}")
print(f"Negative edge scores: {neg_scores_dot.detach().numpy().round(3)}")

# --- MLP decoder ---
mlp = nn.Sequential(
    nn.Linear(h_feats, h_feats),
    nn.ReLU(),
    nn.Dropout(0.0),
    nn.Linear(h_feats, 1),
)

# Hadamard product as input to MLP
pos_scores_mlp = mlp(h[pos_src] * h[pos_dst]).squeeze(-1)
neg_scores_mlp = mlp(h[neg_src] * h[neg_dst]).squeeze(-1)

print(f"\n=== MLP Decoder ===")
print(f"Positive edge scores: {pos_scores_mlp.detach().numpy().round(3)}")
print(f"Negative edge scores: {neg_scores_mlp.detach().numpy().round(3)}")

# Binary cross-entropy loss
all_scores = torch.cat([pos_scores_dot, neg_scores_dot])
all_labels = torch.cat([torch.ones(4), torch.zeros(4)])
loss = nn.functional.binary_cross_entropy_with_logits(all_scores, all_labels)
print(f"\nBCE loss (dot product): {loss.item():.4f}")

## 8. The `num_layers = cg_depth` Constraint

A critical constraint: the GNN's number of layers must equal the computation graph depth (`cg_depth`) stored in the CGT `.pt` file.

**Why?** A computation graph tree of depth $d$ has exactly $d$ hops from leaves to root. A GNN with $L$ layers performs $L$ rounds of message passing. If $L < d$, the root cannot aggregate information from all tree levels — the deepest leaves' features never reach the root. If $L > d$, extra layers perform message passing on a structure where all information has already been fully aggregated to the root (since the tree is acyclic), offering no benefit and risking over-smoothing.

The optimal case is $L = d$: after exactly $d$ rounds of message passing, information from every leaf has propagated to the root through exactly one path.

**Fair comparison across conditions:** If CGT was trained with `cg_depth=2`, all three evaluation conditions must use `num_layers=2`. The Original condition uses a 2-layer GNN on the full graph (2-hop receptive field). The CG conditions use a 2-layer GNN on depth-2 trees. This ensures all conditions have the same receptive field depth, isolating the data source as the only variable.

In [ ]:
# Illustrate why num_layers must equal cg_depth

cg_depth = 2
cg_fanout = 2

# Build a simple tree: root → 2 children → 4 grandchildren
tree = {
    "root": ["c1", "c2"],
    "c1": ["g1", "g2"],
    "c2": ["g3", "g4"],
    "g1": [], "g2": [], "g3": [], "g4": [],
}

# Simulate message passing rounds
initial_info = {node: {node} for node in tree}  # each node knows only itself

def message_pass(info, tree):
    """One round: each node receives info from its children."""
    new_info = {}
    for node, children in tree.items():
        gathered = set(info[node])
        for child in children:
            gathered |= info[child]
        new_info[node] = gathered
    return new_info

info = initial_info.copy()
for layer in range(cg_depth):
    info = message_pass(info, tree)
    root_knows = sorted(info["root"])
    all_nodes = sorted(tree.keys())
    complete = set(root_knows) == set(all_nodes)
    print(f"After layer {layer+1}: root knows {root_knows}")
    print(f"  Complete coverage: {complete}")

print(f"\nWith num_layers={cg_depth} (= cg_depth), root has aggregated")
print(f"information from all {len(tree)} tree nodes ✓")
print(f"\nWith num_layers=1 (< cg_depth), root would only see: "
      f"{sorted(message_pass(initial_info, tree)['root'])}")

## 9. Usage

### Anomaly Detection

```bash
# Via shell script (recommended)
bash scripts/benchmark/run_anomaly_benchmark.sh reddit GCN,GIN,GraphSAGE 3 cgt
# Args: [datasets] [models] [trials] [generator] [synthetic_name] [task]

# With specific synthetic variant and task
bash scripts/benchmark/run_anomaly_benchmark.sh reddit GCN,GIN 1 cgt reddit_e50_k512_c1_d2_f5 hidden_labels

# Directly via Python
python scripts/benchmark/anomaly_benchmark.py \
    --datasets reddit \
    --models GCN,GIN,GraphSAGE \
    --trials 3 \
    --generator cgt \
    --synthetic_type comp-graph \
    --task hidden_labels \
    --num_layers 2
```

### Multi-Trial Evaluation

When per-trial `.pt` files exist (`{variant}_t0.pt` through `{variant}_t9.pt`) inside the variant subdirectory, the benchmark automatically loads a different file per trial so the synthetic train/val/test split varies across trials — matching the split-varying behaviour of the Original and Original-CG conditions. This requires training CGT on all 10 GADBench splits first (see `Methodology/cgt_training.ipynb` Section 8).

When only a single `.pt` file exists (no `_t` suffix), the benchmark falls back to seed-only variation across trials: the same synthetic data is reused, and only the GNN weight initialisation changes.

```bash
# Train all 10 trials first
bash scripts/train/train_cgt.sh reddit 50 512 1 128 2 5 10

# Then evaluate with 10 trials — auto-detects per-trial files in variant subdir
bash scripts/benchmark/run_anomaly_benchmark.sh reddit GCN,GIN,GraphSAGE 10 cgt reddit_e50_k512_c1_d2_f5
```

### Link Prediction

```bash
# Via shell script
bash scripts/benchmark/run_link_benchmark.sh reddit GCN,GIN 3 cgt "" hidden_links random dot
# Args: [datasets] [models] [trials] [generator] [synthetic_name] [task] [neg_sampling] [decoder]

# Hard negatives with MLP decoder
bash scripts/benchmark/run_link_benchmark.sh reddit GCN,GIN 3 cgt "" hidden_links hard mlp

# Directly via Python
python scripts/benchmark/link_benchmark.py \
    --datasets reddit \
    --models GCN,GIN,GraphSAGE \
    --trials 3 \
    --generator cgt \
    --synthetic_type comp-graph \
    --neg_sampling hard \
    --decoder mlp
```

### Key Parameters

| Parameter | Default | Description |
|---|---|---|
| `--datasets` | *(required)* | Comma-separated dataset names |
| `--models` | `GCN,GIN,GraphSAGE,XGBGraph` | GNN architectures to evaluate |
| `--trials` | `1` | Independent evaluation runs |
| `--generator` | *(required)* | Generative model subfolder (`cgt` or `bigg`) |
| `--synthetic_type` | `comp-graph` | Data format (`comp-graph` for CGT, `graph` for BiGG) |
| `--task` | `hidden_labels` | Task subfolder (`hidden_labels`, `hidden_links`, `structure`) |
| `--num_layers` | `2` | GNN layers (**must equal `cg_depth`**) |
| `--batch_size` | `256` | Batch size for CG DataLoaders |
| `--epochs` | `200` | Maximum training epochs |
| `--patience` | `50` | Early stopping patience |
| `--lr` | `0.01` | Adam learning rate |
| `--h_feats` | `32` | GNN hidden dimension |
| `--neg_sampling` | `random` | `random` or `hard` (link prediction only) |
| `--decoder` | `dot` | `dot` or `mlp` (link prediction only) |

### Output

Results are saved to `results/evaluate/evaluation_results.xlsx` (and `.csv`), with columns for source (`original`, `synthetic-cgt`, `original-cg`), dataset, model, and mean/std for each metric across trials.

## References

- Chen, Y., Zhang, Y., Bian, T., Chen, H., Karypis, G. & Li, J. (2023). *Demystifying Graph Condensation with Computation Graphs.* (CGT)
- Esteban, C., Hyland, S. L. & Rätsch, G. (2017). *Real-valued (Medical) Time Series Generation with Recurrent Conditional GANs.* (TSTR paradigm)
- Tang, J. et al. (2023). *GADBench: Revisiting and Benchmarking Supervised Graph Anomaly Detection.* NeurIPS. (Evaluation framework and GNN implementations)
- Yang, Z. et al. (2020). *Understanding Negative Sampling in Graph Representation Learning.* KDD. (Hard negative sampling strategies)